# 모듈 05: 도구를 가진 에이전트(Tool-Calling Agent)

모듈 04에서 MessagesState와 멀티턴 대화 패턴을 익혔습니다.
이번에는 LLM이 스스로 도구 호출 여부를 판단하고, ToolNode가 실제 함수를 실행하는
에이전트 루프 그래프를 구현합니다.

## 이 노트북에서 할 것
| 단계 | 내용 |
|------|------|
| 1 | 에이전트 루프 개념 (agent → tools → agent 순환 구조) |
| 2 | @tool 데코레이터로 도구 정의 + llm.bind_tools()로 LLM에 전달 |
| 3 | ToolNode + tools_condition으로 에이전트 루프 그래프 구성 |
| 4 | 그래프 실행 후 messages 기록으로 도구 호출 과정 확인 |
| 5 | 도구 없는 질문 vs 도구 필요 질문 비교 |

예상 소요 시간: 약 40분  
전제 조건: 모듈 04 완료 (MessagesState, 멀티턴 대화)

## 학습 목표

이 모듈을 완료하면 다음을 할 수 있습니다:

1. `@tool` 데코레이터로 도구 함수를 정의하고 `llm.bind_tools()`로 LLM에
   도구 정보를 전달하는 방법을 설명할 수 있다
2. `ToolNode` + `tools_condition`을 사용하여 `agent → tools → agent` 루프
   구조의 그래프를 구성할 수 있다
3. 실행 후 `messages` 기록에서 `HumanMessage` → `AIMessage(tool_calls)` →
   `ToolMessage` → `AIMessage` 흐름을 확인하고 설명할 수 있다

## 전체 흐름도

```
┌──────────────────────────────────────────────────────┐
│                                                      │
│  [1] 에이전트 루프 개념                              │
│       ↓                                              │
│  [2] 환경 설정 + LLM + 도구 정의 (@tool)             │
│       ↓                                              │
│  [3] ToolNode + tools_condition 그래프 구성          │
│       ↓                                              │
│  [4] 단순 계산 실행 (도구 1회 호출)                  │
│       ↓                                              │
│  [5] 복합 계산 실행 (도구 2회 호출)                  │
│       ↓                                              │
│  [6] messages 기록 분석 + 그래프 시각화              │
│                                                      │
└──────────────────────────────────────────────────────┘
```

## 섹션 1: 에이전트 루프 개념

---

에이전트 루프는 LLM이 도구 호출이 필요하다고 판단할 때 순환하는 구조다:

```
[START]
  ↓
[agent] → LLM 호출 → tool_calls 있음? ─── YES → [tools] ─┐
                                      └── NO  → [END]    │
                                                          │
           ┌───────────────────────────────────────────────┘
           ↓
         [agent] (도구 결과를 포함한 메시지로 다시 LLM 호출)
```

### 일반 엣지 vs 조건부 엣지

| 연결 | 타입 | 설명 |
|------|------|------|
| `agent` → `tools` / `END` | 조건부 | `tools_condition`이 자동 판별 |
| `tools` → `agent` | 일반 | 항상 agent로 돌아옴 |

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import MessagesState, StateGraph, END
from langgraph.prebuilt import ToolNode, tools_condition

notebook_dir = os.path.dirname(os.path.abspath('__file__'))
langgraph_root = os.path.dirname(notebook_dir)
project_root = os.path.dirname(langgraph_root)
langchain_env = os.path.join(project_root, 'LangChain', '.env')
langgraph_env = os.path.join(langgraph_root, '.env')

if os.path.exists(langchain_env):
    load_dotenv(langchain_env)
    print('[OK] LangChain/.env 로드 완료')
elif os.path.exists(langgraph_env):
    load_dotenv(langgraph_env)
    print('[OK] LangGraph/.env 로드 완료')
else:
    print('[FAIL] .env 파일을 찾을 수 없습니다')

api_key = os.getenv('GOOGLE_API_KEY')
print(f'[OK] GOOGLE_API_KEY: {"설정됨" if api_key else "[FAIL] 미설정"}')

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.7)
print('[OK] ChatGoogleGenerativeAI 초기화 완료 (gemini-2.0-flash)')

## 섹션 2: @tool 데코레이터와 bind_tools

---

`@tool` 데코레이터는 일반 파이썬 함수를 LLM이 호출할 수 있는 도구로 변환한다:

```python
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """두 정수를 더한다."""   # docstring이 LLM에게 전달되는 도구 설명
    return a + b
```

`llm.bind_tools(tools)`로 LLM에 도구 목록을 알려준다:

```python
tools = [add, multiply]
llm_with_tools = llm.bind_tools(tools)
# 이후 llm_with_tools.invoke(messages) 호출 시
# LLM이 도구 사용이 필요하다고 판단하면 AIMessage.tool_calls 가 채워짐
```

| 구성 요소 | 설명 |
|-----------|------|
| `@tool` | 함수를 도구로 등록, docstring이 LLM에게 전달되는 도구 설명 |
| `llm.bind_tools(tools)` | LLM에 사용 가능한 도구 목록 전달 |
| `AIMessage.tool_calls` | LLM이 도구 호출을 결정했을 때 채워지는 필드 |

In [ ]:
@tool
def add(a: int, b: int) -> int:
    """두 정수를 더한다."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """두 정수를 곱한다."""
    return a * b

tools = [add, multiply]
llm_with_tools = llm.bind_tools(tools)

print('[OK] 도구 정의 완료')
print(f'  - add: {add.description}')
print(f'  - multiply: {multiply.description}')
print('[OK] llm.bind_tools(tools) 완료 → LLM이 도구 목록을 인식함')

## 섹션 3: ToolNode + tools_condition

---

`ToolNode`는 LLM의 `tool_calls`를 받아 실제 함수를 실행하는 내장 노드다:

```python
from langgraph.prebuilt import ToolNode, tools_condition

tool_node = ToolNode(tools=[add, multiply])
# AIMessage에 tool_calls가 있으면 해당 함수를 실행하고
# 결과를 ToolMessage로 반환
```

`tools_condition`은 `tool_calls` 유무를 자동 판별하는 내장 라우터 함수다:

```python
graph.add_conditional_edges("agent", tools_condition)
# tools_condition이 반환하는 값:
#   - "tools"  : tool_calls가 있을 때
#   - END      : tool_calls가 없을 때 (최종 답변)
```

### 그래프 조립 패턴

```python
graph.add_node("agent", agent)
graph.add_node("tools", tool_node)
graph.set_entry_point("agent")
graph.add_conditional_edges("agent", tools_condition)  # 자동 분기
graph.add_edge("tools", "agent")                       # 도구 결과 → LLM
```

In [ ]:
def agent(state: MessagesState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

tool_node = ToolNode(tools=tools)

agent_graph = StateGraph(MessagesState)
agent_graph.add_node("agent", agent)
agent_graph.add_node("tools", tool_node)
agent_graph.set_entry_point("agent")
agent_graph.add_conditional_edges("agent", tools_condition)
agent_graph.add_edge("tools", "agent")
agent_app = agent_graph.compile()

print('[OK] 에이전트 루프 그래프 컴파일 완료')
print('  노드: agent, tools')
print('  엣지: START→agent, agent→(tools_condition)→tools/END, tools→agent')

## 섹션 4: 에이전트 루프 실행

---

그래프를 실행하면 LLM이 도구 호출 여부를 스스로 판단한다.
도구 호출이 필요한 질문을 입력하면 `agent → tools → agent` 순으로 순환한다:

```
입력: "3 더하기 5는?"
  1. [agent]  → LLM: "add(3, 5) 를 호출해야겠다" → AIMessage(tool_calls=[...])
  2. [tools]  → add(3, 5) 실행 → ToolMessage(content="8")
  3. [agent]  → LLM: "결과는 8입니다" → AIMessage(content="3 더하기 5는 8입니다.")
  4. [END]    → 최종 답변 반환
```

도구 호출이 불필요한 질문은 `agent → END`로 바로 종료된다.

In [ ]:
print('=== 단순 계산: 3 + 5 ===')
result1 = agent_app.invoke({
    "messages": [HumanMessage(content="3 더하기 5는 얼마야?")]
})

print(f'사용자: 3 더하기 5는 얼마야?')
print(f'메시지 기록 총 {len(result1["messages"])}개:')
for i, msg in enumerate(result1["messages"], 1):
    if isinstance(msg, HumanMessage):
        print(f'  {i}. [사용자] {msg.content}')
    elif isinstance(msg, AIMessage):
        if msg.tool_calls:
            print(f'  {i}. [AI→도구] tool_calls: {[tc["name"] for tc in msg.tool_calls]}')
        else:
            print(f'  {i}. [AI 최종] {msg.content}')
    elif isinstance(msg, ToolMessage):
        print(f'  {i}. [도구결과] {msg.content}')

print(f'\n[OK] 최종 답변: {result1["messages"][-1].content}')

In [ ]:
print('=== 복합 계산: 2 × 4 + 7 ===')
result2 = agent_app.invoke({
    "messages": [HumanMessage(content="2 곱하기 4를 계산하고, 그 결과에 7을 더해줘")]
})

print(f'사용자: 2 곱하기 4를 계산하고, 그 결과에 7을 더해줘')
print(f'메시지 기록 총 {len(result2["messages"])}개:')
for i, msg in enumerate(result2["messages"], 1):
    if isinstance(msg, HumanMessage):
        print(f'  {i}. [사용자] {msg.content}')
    elif isinstance(msg, AIMessage):
        if msg.tool_calls:
            print(f'  {i}. [AI→도구] tool_calls: {[tc["name"] for tc in msg.tool_calls]}')
        else:
            print(f'  {i}. [AI 최종] {msg.content}')
    elif isinstance(msg, ToolMessage):
        print(f'  {i}. [도구결과] {msg.content}')

print(f'\n[OK] 최종 답변: {result2["messages"][-1].content}')

## 섹션 5: messages 기록으로 도구 호출 과정 확인

---

실행 후 `messages` 기록에는 다음 4가지 타입의 메시지가 교대로 쌓인다:

| 순서 | 타입 | 설명 |
|------|------|------|
| 1 | `HumanMessage` | 사용자 입력 |
| 2 | `AIMessage(tool_calls=[...])` | LLM이 도구 호출을 결정 (tool_calls 채워짐) |
| 3 | `ToolMessage` | 도구 실행 결과 (ToolNode가 추가) |
| 4 | `AIMessage(content="...")` | LLM이 도구 결과를 보고 최종 답변 생성 |

도구가 필요 없는 질문의 경우 2단계(`AIMessage`)에서 바로 종료된다.

In [ ]:
print('=== 도구 없는 질문 (직접 답변) ===')
result3 = agent_app.invoke({
    "messages": [HumanMessage(content="파이썬이 무엇인지 한 문장으로 설명해줘")]
})
print(f'메시지 수: {len(result3["messages"])} (도구 호출 없음)')
print(f'AI: {result3["messages"][-1].content[:80]}...')

print()
print('=== 도구 필요 질문 (에이전트 루프) ===')
print(f'앞서 실행한 result1 메시지 수: {len(result1["messages"])} (도구 1회 호출)')
print(f'앞서 실행한 result2 메시지 수: {len(result2["messages"])} (도구 2회 이상 호출)')

print()
print('[OK] 메시지 수로 에이전트 루프 횟수를 추정할 수 있습니다')

## 그래프 구조 시각화

---

`get_graph().print_ascii()`로 에이전트 루프 그래프 구조를 확인합니다.  
모듈 04의 선형 구조(`chatbot → END`)와 달리 `tools → agent` 역방향 엣지가 추가되어
루프 구조가 형성된 것을 확인할 수 있습니다.

In [ ]:
print('=== 에이전트 루프 그래프 ===')
agent_app.get_graph().print_ascii()

## 마무리

---

### 핵심 패턴 요약

```python
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from langgraph.graph import MessagesState, StateGraph, END
from langgraph.prebuilt import ToolNode, tools_condition

# 1. 도구 정의
@tool
def add(a: int, b: int) -> int:
    """두 정수를 더한다."""
    return a + b

tools = [add]
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.7)
llm_with_tools = llm.bind_tools(tools)

# 2. 에이전트 노드
def agent(state: MessagesState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# 3. 그래프 조립
graph = StateGraph(MessagesState)
graph.add_node("agent", agent)
graph.add_node("tools", ToolNode(tools=tools))
graph.set_entry_point("agent")
graph.add_conditional_edges("agent", tools_condition)
graph.add_edge("tools", "agent")
app = graph.compile()

# 4. 실행
result = app.invoke({"messages": [HumanMessage(content="3 더하기 5는?")]})
```

### 핵심 API 레퍼런스

| API | 설명 |
|-----|------|
| `@tool` | 함수를 LLM 호출 가능한 도구로 등록 |
| `llm.bind_tools(tools)` | LLM에 도구 목록 전달 |
| `ToolNode(tools=[...])` | tool_calls를 받아 실제 함수 실행하는 노드 |
| `tools_condition` | tool_calls 유무를 자동 판별하는 내장 라우터 |
| `AIMessage.tool_calls` | LLM이 도구 호출 결정 시 채워지는 필드 |
| `ToolMessage` | 도구 실행 결과 메시지 타입 |

## LangGraph 커리큘럼 마무리 + 자기 점검

---

모듈 00부터 05까지 LangGraph 핵심 개념을 모두 학습했습니다.

### 지금까지 배운 것

| 모듈 | 핵심 개념 |
|------|----------|
| 00 | langgraph 설치, API 키 설정 |
| 01 | StateGraph, 노드·엣지·컴파일·실행 4단계 |
| 02 | TypedDict 상태, 노드 입출력 규칙, 리듀서 |
| 03 | 조건부 엣지, 라우터 함수, add_conditional_edges |
| 04 | MessagesState, add_messages, 멀티턴 대화 |
| 05 | @tool, bind_tools, ToolNode, tools_condition, 에이전트 루프 |

### 자기 점검 체크리스트

- [ ] `@tool` 데코레이터로 도구를 정의하고 `bind_tools()`로 LLM에 전달할 수 있나요?
- [ ] `ToolNode` + `tools_condition`으로 에이전트 루프 그래프를 구성할 수 있나요?
- [ ] 실행 후 `messages` 기록에서 HumanMessage → AIMessage(tool_calls) → ToolMessage → AIMessage 흐름을 읽을 수 있나요?